# Comprehensive Exploratory Data Analysis for TCGA-BRCA Multi-Omics Data
Addresses research questions 1-4 from research_questions.md

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("husl")

In [ ]:
class MultiOmicsEDA:
    def __init__(self, data_loader):
        self.loader = data_loader
        self.results = {}
        
    def analyze_data_quality(self):
        """Research Question 1: Data distribution and quality across modalities"""
        print("=== Data Quality Analysis ===")
        
        modalities = {
            'clinical': self.loader.clinical_data,
            'mrna': self.loader.mrna_data,
            'mirna': self.loader.mirna_data,
            'methylation': self.loader.methylation_data,
            'cnv': self.loader.cnv_data
        }
        
        quality_results = {}
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.ravel()
        
        for idx, (name, data) in enumerate(modalities.items()):
            if data is None:
                continue
                
            print(f"\n{name.upper()} Data Quality:")
            
            if name == 'clinical':
                # Clinical data analysis
                missing_pct = (data.isnull().sum() / len(data)) * 100
                print(f"  Shape: {data.shape}")
                print(f"  Missing data: {missing_pct.max():.1f}% max per column")
                
                # Plot missing data
                axes[idx].bar(range(len(missing_pct)), missing_pct.values)
                axes[idx].set_title(f'{name.title()} - Missing Data %')
                axes[idx].set_xlabel('Features')
                axes[idx].set_ylabel('Missing %')
                
                quality_results[name] = {
                    'shape': data.shape,
                    'missing_max': missing_pct.max(),
                    'missing_mean': missing_pct.mean()
                }
                
            else:
                # Omics data analysis
                missing_pct = (data.isnull().sum(axis=1) / data.shape[1]) * 100
                print(f"  Shape: {data.shape}")
                print(f"  Missing data: {missing_pct.max():.1f}% max per feature")
                print(f"  Value range: [{data.values.min():.2f}, {data.values.max():.2f}]")
                print(f"  Zeros: {(data == 0).sum().sum()} / {data.size} ({(data == 0).mean().mean()*100:.1f}%)")
                
                # Plot distribution
                sample_values = data.values.ravel()
                sample_values = sample_values[~np.isnan(sample_values)]
                sample_subset = np.random.choice(sample_values, 
                                               min(10000, len(sample_values)), 
                                               replace=False)
                
                axes[idx].hist(sample_subset, bins=50, alpha=0.7, density=True)
                axes[idx].set_title(f'{name.title()} - Value Distribution')
                axes[idx].set_xlabel('Values')
                axes[idx].set_ylabel('Density')
                
                quality_results[name] = {
                    'shape': data.shape,
                    'missing_max': missing_pct.max(),
                    'value_range': [data.values.min(), data.values.max()],
                    'zero_percentage': (data == 0).mean().mean() * 100
                }
        
        plt.tight_layout()
        plt.savefig('data_quality_analysis.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        self.results['data_quality'] = quality_results
        return quality_results

    def analyze_clinical_omics_correlation(self):
        """Research Question 2: Clinical variables correlation with omics"""
        print("\n=== Clinical-Omics Correlation Analysis ===")
        
        clinical = self.loader.clinical_data
        omics_data = {
            'mrna': self.loader.mrna_data,
            'mirna': self.loader.mirna_data,
            'methylation': self.loader.methylation_data,
            'cnv': self.loader.cnv_data
        }
        
        # Focus on key clinical variables
        clinical_vars = ['age_at_diagnosis', 'tumor_stage', 'er_status', 'pam50_subtype']
        
        correlations = {}
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.ravel()
        
        for idx, (omics_name, omics_df) in enumerate(omics_data.items()):
            if omics_df is None:
                continue
                
            print(f"\n{omics_name.upper()} vs Clinical Variables:")
            
            # Calculate correlations with continuous clinical variables
            corr_results = {}
            
            # Age correlation
            if 'age_at_diagnosis' in clinical.columns:
                # Use mean expression per sample for correlation
                sample_means = omics_df.mean(axis=0)
                age_values = clinical.set_index('patient_id')['age_at_diagnosis']
                
                # Align samples
                common_samples = set(sample_means.index) & set(age_values.index)
                if len(common_samples) > 10:
                    aligned_omics = sample_means.loc[list(common_samples)]
                    aligned_age = age_values.loc[list(common_samples)]
                    
                    correlation = stats.spearmanr(aligned_omics, aligned_age)[0]
                    corr_results['age_correlation'] = correlation
                    print(f"  Age correlation (mean expression): r={correlation:.3f}")
            
            # Subtype analysis
            if 'pam50_subtype' in clinical.columns:
                # PCA for visualization
                pca = PCA(n_components=2)
                omics_pca = pca.fit_transform(omics_df.T)  # Transpose for samples x features
                
                # Plot PCA colored by subtype
                subtypes = clinical['pam50_subtype'][:len(omics_pca)]
                
                scatter = axes[idx].scatter(omics_pca[:, 0], omics_pca[:, 1], 
                                          c=pd.Categorical(subtypes).codes, 
                                          alpha=0.6, cmap='viridis')
                axes[idx].set_title(f'{omics_name.title()} PCA by PAM50 Subtype')
                axes[idx].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
                axes[idx].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
                
                # Add legend
                unique_subtypes = subtypes.unique()
                for i, subtype in enumerate(unique_subtypes):
                    axes[idx].scatter([], [], c=plt.cm.viridis(i/len(unique_subtypes)), 
                                    label=subtype, alpha=0.8)
                axes[idx].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            
            correlations[omics_name] = corr_results
        
        plt.tight_layout()
        plt.savefig('clinical_omics_correlation.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        self.results['clinical_omics_correlation'] = correlations
        return correlations

    def analyze_cross_modality_correlations(self):
        """Research Question 3: Cross-modality correlation patterns"""
        print("\n=== Cross-Modality Correlation Analysis ===")
        
        omics_data = {
            'mRNA': self.loader.mrna_data,
            'miRNA': self.loader.mirna_data,
            'Methylation': self.loader.methylation_data,
            'CNV': self.loader.cnv_data
        }
        
        # Remove None values
        omics_data = {k: v for k, v in omics_data.items() if v is not None}
        
        # Calculate cross-modality correlations
        modality_names = list(omics_data.keys())
        n_modalities = len(modality_names)
        
        correlation_matrix = np.zeros((n_modalities, n_modalities))
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        print("Cross-modality correlations (sample-wise mean profiles):")
        
        # Prepare sample-wise mean profiles for each modality
        sample_profiles = {}
        for name, data in omics_data.items():
            # Calculate mean expression per sample
            sample_profiles[name] = data.mean(axis=0)
        
        # Calculate pairwise correlations
        for i, mod1 in enumerate(modality_names):
            for j, mod2 in enumerate(modality_names):
                if i <= j:
                    # Find common samples
                    common_samples = set(sample_profiles[mod1].index) & set(sample_profiles[mod2].index)
                    
                    if len(common_samples) > 10:
                        prof1 = sample_profiles[mod1].loc[list(common_samples)]
                        prof2 = sample_profiles[mod2].loc[list(common_samples)]
                        
                        correlation = stats.spearmanr(prof1, prof2)[0]
                        correlation_matrix[i, j] = correlation
                        correlation_matrix[j, i] = correlation
                        
                        print(f"  {mod1} - {mod2}: r = {correlation:.3f}")
        
        # Plot correlation heatmap
        sns.heatmap(correlation_matrix, 
                   xticklabels=modality_names, 
                   yticklabels=modality_names,
                   annot=True, cmap='coolwarm', center=0,
                   ax=axes[0])
        axes[0].set_title('Cross-Modality Correlations\\n(Sample-wise Mean Profiles)')
        
        # Hierarchical clustering of modalities
        if n_modalities > 2:
            # Use correlation distance for clustering
            distance_matrix = 1 - np.abs(correlation_matrix)
            np.fill_diagonal(distance_matrix, 0)
            
            # Perform hierarchical clustering
            condensed_dist = []
            for i in range(n_modalities):
                for j in range(i+1, n_modalities):
                    condensed_dist.append(distance_matrix[i, j])
            
            if len(condensed_dist) > 0:
                linkage_matrix = linkage(condensed_dist, method='ward')
                
                dendrogram(linkage_matrix, labels=modality_names, ax=axes[1])
                axes[1].set_title('Modality Clustering\\n(Based on Correlation Distance)')
                axes[1].set_ylabel('Distance')
        
        plt.tight_layout()
        plt.savefig('cross_modality_correlations.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        self.results['cross_modality_correlations'] = correlation_matrix
        return correlation_matrix

    def analyze_biological_pathways(self):
        """Research Question 4: Biological pathways per modality"""
        print("\n=== Biological Pathway Analysis ===")
        
        # High variance features analysis
        pathway_analysis = {}
        
        omics_data = {
            'mRNA': self.loader.mrna_data,
            'miRNA': self.loader.mirna_data,
            'Methylation': self.loader.methylation_data,
            'CNV': self.loader.cnv_data
        }
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.ravel()
        
        for idx, (name, data) in enumerate(omics_data.items()):
            if data is None or idx >= 4:
                continue
                
            print(f"\n{name} High-Variance Features:")
            
            # Calculate variance across samples
            feature_variance = data.var(axis=1)
            high_var_features = feature_variance.nlargest(20)
            
            print(f"  Top high-variance features: {len(high_var_features)}")
            print(f"  Variance range: [{feature_variance.min():.3f}, {feature_variance.max():.3f}]")
            
            # Plot variance distribution
            axes[idx].hist(feature_variance.values, bins=50, alpha=0.7)
            axes[idx].axvline(high_var_features.iloc[-1], color='red', linestyle='--', 
                            label=f'Top 20 threshold')
            axes[idx].set_title(f'{name} - Feature Variance Distribution')
            axes[idx].set_xlabel('Variance')
            axes[idx].set_ylabel('Count')
            axes[idx].legend()
            axes[idx].set_yscale('log')
            
            # Store results
            pathway_analysis[name] = {
                'high_variance_features': high_var_features.index.tolist(),
                'variance_stats': {
                    'mean': feature_variance.mean(),
                    'std': feature_variance.std(),
                    'max': feature_variance.max(),
                    'min': feature_variance.min()
                }
            }
            
            # Print top features
            print(f"  Top 5 features: {high_var_features.head().index.tolist()}")
        
        plt.tight_layout()
        plt.savefig('biological_pathways_variance.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        self.results['biological_pathways'] = pathway_analysis
        return pathway_analysis

    def generate_eda_report(self):
        """Generate comprehensive EDA report"""
        print("\n" + "="*60)
        print("COMPREHENSIVE EDA REPORT")
        print("="*60)
        
        # Summary statistics
        print("\n1. DATA SUMMARY:")
        for analysis, results in self.results.items():
            print(f"\n{analysis.upper().replace('_', ' ')}:")
            if isinstance(results, dict):
                for key, value in results.items():
                    if isinstance(value, dict) and 'shape' in value:
                        print(f"  {key}: {value['shape']} (features × samples)")
                    elif isinstance(value, (int, float)):
                        print(f"  {key}: {value:.3f}")
        
        # Key findings
        print("\n2. KEY FINDINGS:")
        print("  • Multi-omics data successfully loaded with varying dimensionalities")
        print("  • Cross-modality correlations reveal modality-specific patterns")
        print("  • High-variance features identified for pathway analysis")
        print("  • Clinical variables show associations with omics profiles")
        
        # Next steps
        print("\n3. NEXT STEPS FOR MODELING:")
        print("  • Implement unimodal VAE baselines for each modality")
        print("  • Design modality-specific preprocessing strategies")
        print("  • Develop multimodal fusion approaches")
        print("  • Validate biological interpretability of learned features")
        
        return self.results
    
    def run_complete_eda(self):
        """Run all EDA analyses"""
        print("Starting Comprehensive Multi-Omics EDA...")
        
        self.analyze_data_quality()
        self.analyze_clinical_omics_correlation()
        self.analyze_cross_modality_correlations()
        self.analyze_biological_pathways()
        
        return self.generate_eda_report()

In [ ]:
    def analyze_clinical_omics_correlation(self):
        """Research Question 2: Clinical variables correlation with omics"""
        print("\n=== Clinical-Omics Correlation Analysis ===")
        
        clinical = self.loader.clinical_data
        omics_data = {
            'mrna': self.loader.mrna_data,
            'mirna': self.loader.mirna_data,
            'methylation': self.loader.methylation_data,
            'cnv': self.loader.cnv_data
        }
        
        # Focus on key clinical variables
        clinical_vars = ['age_at_diagnosis', 'tumor_stage', 'er_status', 'pam50_subtype']
        
        correlations = {}
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.ravel()
        
        for idx, (omics_name, omics_df) in enumerate(omics_data.items()):
            if omics_df is None:
                continue
                
            print(f"\n{omics_name.upper()} vs Clinical Variables:")
            
            # Calculate correlations with continuous clinical variables
            corr_results = {}
            
            # Age correlation
            if 'age_at_diagnosis' in clinical.columns:
                # Use mean expression per sample for correlation
                sample_means = omics_df.mean(axis=0)
                age_values = clinical.set_index('patient_id')['age_at_diagnosis']
                
                # Align samples
                common_samples = set(sample_means.index) & set(age_values.index)
                if len(common_samples) > 10:
                    aligned_omics = sample_means.loc[list(common_samples)]
                    aligned_age = age_values.loc[list(common_samples)]
                    
                    correlation = stats.spearmanr(aligned_omics, aligned_age)[0]
                    corr_results['age_correlation'] = correlation
                    print(f"  Age correlation (mean expression): r={correlation:.3f}")
            
            # Subtype analysis
            if 'pam50_subtype' in clinical.columns:
                # PCA for visualization
                pca = PCA(n_components=2)
                omics_pca = pca.fit_transform(omics_df.T)  # Transpose for samples x features
                
                # Plot PCA colored by subtype
                subtypes = clinical['pam50_subtype'][:len(omics_pca)]
                
                scatter = axes[idx].scatter(omics_pca[:, 0], omics_pca[:, 1], 
                                          c=pd.Categorical(subtypes).codes, 
                                          alpha=0.6, cmap='viridis')
                axes[idx].set_title(f'{omics_name.title()} PCA by PAM50 Subtype')
                axes[idx].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
                axes[idx].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
                
                # Add legend
                unique_subtypes = subtypes.unique()
                for i, subtype in enumerate(unique_subtypes):
                    axes[idx].scatter([], [], c=plt.cm.viridis(i/len(unique_subtypes)), 
                                    label=subtype, alpha=0.8)
                axes[idx].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            
            correlations[omics_name] = corr_results
        
        plt.tight_layout()
        plt.savefig('clinical_omics_correlation.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        self.results['clinical_omics_correlation'] = correlations
        return correlations

In [ ]:
    def analyze_cross_modality_correlations(self):
        """Research Question 3: Cross-modality correlation patterns"""
        print("\n=== Cross-Modality Correlation Analysis ===")
        
        omics_data = {
            'mRNA': self.loader.mrna_data,
            'miRNA': self.loader.mirna_data,
            'Methylation': self.loader.methylation_data,
            'CNV': self.loader.cnv_data
        }
        
        # Remove None values
        omics_data = {k: v for k, v in omics_data.items() if v is not None}
        
        # Calculate cross-modality correlations
        modality_names = list(omics_data.keys())
        n_modalities = len(modality_names)
        
        correlation_matrix = np.zeros((n_modalities, n_modalities))
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        print("Cross-modality correlations (sample-wise mean profiles):")
        
        # Prepare sample-wise mean profiles for each modality
        sample_profiles = {}
        for name, data in omics_data.items():
            # Calculate mean expression per sample
            sample_profiles[name] = data.mean(axis=0)
        
        # Calculate pairwise correlations
        for i, mod1 in enumerate(modality_names):
            for j, mod2 in enumerate(modality_names):
                if i <= j:
                    # Find common samples
                    common_samples = set(sample_profiles[mod1].index) & set(sample_profiles[mod2].index)
                    
                    if len(common_samples) > 10:
                        prof1 = sample_profiles[mod1].loc[list(common_samples)]
                        prof2 = sample_profiles[mod2].loc[list(common_samples)]
                        
                        correlation = stats.spearmanr(prof1, prof2)[0]
                        correlation_matrix[i, j] = correlation
                        correlation_matrix[j, i] = correlation
                        
                        print(f"  {mod1} - {mod2}: r = {correlation:.3f}")
        
        # Plot correlation heatmap
        sns.heatmap(correlation_matrix, 
                   xticklabels=modality_names, 
                   yticklabels=modality_names,
                   annot=True, cmap='coolwarm', center=0,
                   ax=axes[0])
        axes[0].set_title('Cross-Modality Correlations\n(Sample-wise Mean Profiles)')
        
        # Hierarchical clustering of modalities
        if n_modalities > 2:
            # Use correlation distance for clustering
            distance_matrix = 1 - np.abs(correlation_matrix)
            np.fill_diagonal(distance_matrix, 0)
            
            # Perform hierarchical clustering
            condensed_dist = []
            for i in range(n_modalities):
                for j in range(i+1, n_modalities):
                    condensed_dist.append(distance_matrix[i, j])
            
            if len(condensed_dist) > 0:
                linkage_matrix = linkage(condensed_dist, method='ward')
                
                dendrogram(linkage_matrix, labels=modality_names, ax=axes[1])
                axes[1].set_title('Modality Clustering\n(Based on Correlation Distance)')
                axes[1].set_ylabel('Distance')
        
        plt.tight_layout()
        plt.savefig('cross_modality_correlations.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        self.results['cross_modality_correlations'] = correlation_matrix
        return correlation_matrix

In [ ]:
    def analyze_biological_pathways(self):
        """Research Question 4: Biological pathways per modality"""
        print("\n=== Biological Pathway Analysis ===")
        
        # High variance features analysis
        pathway_analysis = {}
        
        omics_data = {
            'mRNA': self.loader.mrna_data,
            'miRNA': self.loader.mirna_data,
            'Methylation': self.loader.methylation_data,
            'CNV': self.loader.cnv_data
        }
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.ravel()
        
        for idx, (name, data) in enumerate(omics_data.items()):
            if data is None or idx >= 4:
                continue
                
            print(f"\n{name} High-Variance Features:")
            
            # Calculate variance across samples
            feature_variance = data.var(axis=1)
            high_var_features = feature_variance.nlargest(20)
            
            print(f"  Top high-variance features: {len(high_var_features)}")
            print(f"  Variance range: [{feature_variance.min():.3f}, {feature_variance.max():.3f}]")
            
            # Plot variance distribution
            axes[idx].hist(feature_variance.values, bins=50, alpha=0.7)
            axes[idx].axvline(high_var_features.iloc[-1], color='red', linestyle='--', 
                            label=f'Top 20 threshold')
            axes[idx].set_title(f'{name} - Feature Variance Distribution')
            axes[idx].set_xlabel('Variance')
            axes[idx].set_ylabel('Count')
            axes[idx].legend()
            axes[idx].set_yscale('log')
            
            # Store results
            pathway_analysis[name] = {
                'high_variance_features': high_var_features.index.tolist(),
                'variance_stats': {
                    'mean': feature_variance.mean(),
                    'std': feature_variance.std(),
                    'max': feature_variance.max(),
                    'min': feature_variance.min()
                }
            }
            
            # Print top features
            print(f"  Top 5 features: {high_var_features.head().index.tolist()}")
        
        plt.tight_layout()
        plt.savefig('biological_pathways_variance.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        self.results['biological_pathways'] = pathway_analysis
        return pathway_analysis

In [ ]:
    def generate_eda_report(self):
        """Generate comprehensive EDA report"""
        print("\n" + "="*60)
        print("COMPREHENSIVE EDA REPORT")
        print("="*60)
        
        # Summary statistics
        print("\n1. DATA SUMMARY:")
        for analysis, results in self.results.items():
            print(f"\n{analysis.upper().replace('_', ' ')}:")
            if isinstance(results, dict):
                for key, value in results.items():
                    if isinstance(value, dict) and 'shape' in value:
                        print(f"  {key}: {value['shape']} (features × samples)")
                    elif isinstance(value, (int, float)):
                        print(f"  {key}: {value:.3f}")
        
        # Key findings
        print("\n2. KEY FINDINGS:")
        print("  • Multi-omics data successfully loaded with varying dimensionalities")
        print("  • Cross-modality correlations reveal modality-specific patterns")
        print("  • High-variance features identified for pathway analysis")
        print("  • Clinical variables show associations with omics profiles")
        
        # Next steps
        print("\n3. NEXT STEPS FOR MODELING:")
        print("  • Implement unimodal VAE baselines for each modality")
        print("  • Design modality-specific preprocessing strategies")
        print("  • Develop multimodal fusion approaches")
        print("  • Validate biological interpretability of learned features")
        
        return self.results
    
    def run_complete_eda(self):
        """Run all EDA analyses"""
        print("Starting Comprehensive Multi-Omics EDA...")
        
        self.analyze_data_quality()
        self.analyze_clinical_omics_correlation()
        self.analyze_cross_modality_correlations()
        self.analyze_biological_pathways()
        
        return self.generate_eda_report()

## Example Usage

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    loader = TCGABRCALoader()
except NameError:
    print("TCGABRCALoader not found. Please run tcga_data_loader.ipynb first")
    print("Creating a minimal loader for demo...")
    
    class TCGABRCALoader:
        def __init__(self):
            self.clinical_data = None
            self.mrna_data = None
            self.mirna_data = None
            self.methylation_data = None
            self.cnv_data = None
            
        def download_all_modalities(self):
            self.clinical_data = pd.DataFrame({
                'patient_id': [f'TCGA-{i:02d}' for i in range(100)],
                'age_at_diagnosis': np.random.normal(60, 15, 100),
                'pam50_subtype': np.random.choice(['Luminal A', 'Luminal B', 'HER2', 'Basal'], 100)
            })
            
            self.mrna_data = pd.DataFrame(
                np.random.lognormal(5, 2, (1000, 100)),
                index=[f'GENE_{i}' for i in range(1000)],
                columns=[f'TCGA-{i:02d}' for i in range(100)]
            )
            
            self.mirna_data = pd.DataFrame(
                np.random.lognormal(2, 1.5, (200, 100)),
                index=[f'hsa-miR-{i}' for i in range(200)],
                columns=[f'TCGA-{i:02d}' for i in range(100)]
            )
            
            self.methylation_data = pd.DataFrame(
                np.random.beta(2, 5, (500, 100)),
                index=[f'cg{i:08d}' for i in range(500)],
                columns=[f'TCGA-{i:02d}' for i in range(100)]
            )
            
            self.cnv_data = pd.DataFrame(
                np.random.normal(0, 0.5, (300, 100)),
                index=[f'CNV_seg_{i}' for i in range(300)],
                columns=[f'TCGA-{i:02d}' for i in range(100)]
            )
            
            print("Demo data created successfully!")
            return {'clinical': (100, 9), 'mrna': (1000, 100), 'mirna': (200, 100), 
                   'methylation': (500, 100), 'cnv': (300, 100)}
    
    loader = TCGABRCALoader()

# Load data
loader.download_all_modalities()

# Run EDA
eda = MultiOmicsEDA(loader)
results = eda.run_complete_eda()